# 1. 制氢数据处理

## 1.1 制氢文件拼接

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------------
# 0) 配置
# -----------------------------------
filter_value = 10
# keys = ['main', 'HighPEM', 'LowAE', 'LowAE&HighPEM','LowPV','LowPV&HighPEM','LowPVAE','LowPV+AE&HighPEM',
#         'RegionalRE','RegionalRE_HighPEM','RegionalRE_LowAE','RegionalRE_LowAE&HighPEM','']
keys = ['RegionRE_2030', 'RegionRE_LowAE', 'RegionRE_LowAE_2040', 'RegionRE_LowAE_2050']

variables = [
    'levelized_hydrogen_production_cost_per_kg',
    'wind_capacity',
    'pv_capacity',
    'battery_energy_capacity',
    'battery_storage_capacity',
    'electrolyzer_capacity_AE',
    'electrolyzer_capacity_PEM',
    'electrolyzer_utilization_hours_percent',
    'region'
]

# -----------------------------------
# 1) 处理函数：经纬度映射 + 成本过滤
# -----------------------------------
def process_df(df: pd.DataFrame, minute: int) -> pd.DataFrame:
    # latitude: 90 - 0.25 * orig
    if 'latitude' in df.columns:
        df['latitude'] = 90 - df['latitude'] * (180 / 720)
    # longitude: (hour*60 + orig)/4, hour>=12 再 -360
    if 'longitude' in df.columns:
        hour = minute // 60
        df['longitude'] = (hour * 60 + df['longitude']) / 4 - (360 if hour >= 12 else 0)
    # cost>阈值 → 除经纬度外全列置 0
    col_cost = 'levelized_hydrogen_production_cost_per_kg'
    if col_cost in df.columns:
        other_cols = [c for c in df.columns if c not in ['latitude','longitude']]
        mask = df[col_cost] > filter_value
        df.loc[mask, other_cols] = 0
    return df

# -----------------------------------
# 2) 读入并拼接所有文件
# -----------------------------------
# 用列表暂存，每个 key 最后 concat
df_all = {k: [] for k in keys}

for minute in range(0, 1440, 60):
    for k in keys:
        # 基础文件
        suf = '' if k == 'main' else f'_{k}'
        path = f'DataOutput/hydrogen_results_{minute}{suf}.csv'
        if os.path.exists(path):
            df = pd.read_csv(path)
            df_all[k].append(process_df(df.copy(), minute))

        # _1_ 后缀（除 main 外）
        if k != 'main':
            p1 = f'DataOutput/hydrogen_results_{minute}_1_{k}.csv'
            if os.path.exists(p1):
                df = pd.read_csv(p1)
                df_all[k].append(process_df(df.copy(), minute))
        
        # _1 后缀
        if k == 'main':
            p1 = f'DataOutput/hydrogen_results_{minute}_1.csv'
            if os.path.exists(p1):
                df = pd.read_csv(p1)
                df_all[k].append(process_df(df.copy(), minute))
                
#         # _append 后缀
#         if k == 'main':
#             pa = f'DataOutput/hydrogen_production_{minute}_append.csv'
#         else:
#             pa = f'DataOutput/hydrogen_production_{minute}_{k}_append.csv'
#         if os.path.exists(pa):
#             df = pd.read_csv(pa)
#             df_all[k].append(process_df(df.copy(), minute))

#         if os.path.exists(pa):
#             df = pd.read_csv(pa)
#             df_all[k].append(process_df(df.copy(), minute))

# 把列表 concat 成单个 DataFrame
df_combined = {}
for k in keys:
    if df_all[k]:
        df_combined[k] = pd.concat(df_all[k], ignore_index=True)
    else:
        # If no files were found for this key, create an empty DataFrame
        # with the expected columns to avoid errors downstream.
        print(f"Warning: No data found for key '{k}'. Creating an empty DataFrame.")
        df_combined[k] = pd.DataFrame(columns=variables + ['latitude', 'longitude'])


# -----------------------------------
# 3) 跨时段聚合 (按经纬度平均)
# -----------------------------------
for k, df in df_combined.items():
    # Skip aggregation if the DataFrame is empty
    if not df.empty and {'latitude','longitude'}.issubset(df.columns):
        # Ensure 'region' is handled correctly during mean() calculation
        numeric_vars = [v for v in variables if v != 'region']
        
        # Aggregate numeric variables
        df_agg = (
            df
            .groupby(['latitude','longitude'], as_index=False)[numeric_vars]
            .mean()
        )
        
        # Handle 'region' separately, taking the first non-null value
        if 'region' in df.columns:
            df_region = (
                df.dropna(subset=['region'])
                .groupby(['latitude','longitude'], as_index=False)['region']
                .first()
            )
            # Merge region back into the aggregated dataframe
            df_combined[k] = pd.merge(df_agg, df_region, on=['latitude', 'longitude'], how='left')
        else:
            df_combined[k] = df_agg

# -----------------------------------
# 4) 构造完整 0.25°×0.25° 网格并补全
# -----------------------------------
lat_vals = np.arange(90, -90 - 0.25, -0.25)
lon_vals = np.arange(-180, 180 + 0.25, 0.25)
grid = (
    pd.MultiIndex
      .from_product([lat_vals, lon_vals], names=['latitude','longitude'])
      .to_frame(index=False)
)

for k in keys:
    df = df_combined[k]
    # left join → 缺失填 0
    df_full = (
        grid
        .merge(df, on=['latitude','longitude'], how='left')
        .fillna(0)
    )
    df_combined[k] = df_full
    df_full.to_csv(f'DataOutput/combined_{k}_hydrogen_production_results.csv', index=False)
    print(f"Saved combined_{k}_…")


## 1.2 制氢结果绘图（制氢成本）

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# -----------------------------------
# 边界矢量数据读取
# -----------------------------------
gdf = gpd.read_file("ne_110m_admin_0_countries.shp")
world_4326 = gdf.to_crs("EPSG:4326")

# -----------------------------------
# region 栅格 & colormap 配置（同你已有的）
# -----------------------------------
tif_path = "F:/Workspace/World_Region_025d.tif"
highlight_regions = [0,2,7,9,10,11,12,13,15,18,24,27,28,31]
highlight_colors  = [
    '#e41a1c','#377eb8','#4daf4a','#984ea3',
    '#ff7f00','#ffff33','#a65628','#f781bf','#00008b',
    '#999999','#66c2a5','#fc8d62','#8da0cb','#e78ac3'
]
other_region_color = '#DEEBF7'
region_names = [
    "EU-15","Canada","USA","Central America and Caribbean","Mexico",
    "Colombia","Southeast Asia","Australia_NZ","#N/A","Africa_Northern",
    "Africa_Western","Africa_Eastern","Africa_Southern","South Africa",
    "South America_Northern","South America_Southern","Brazil","Argentina",
    "European Free Trade Association","Russia","EU-12","Europe_Eastern",
    "Europe_Non_EU","Central Asia","Middle East","South Asia","Pakistan",
    "Japan","South Korea","India","Indonesia","China"
]  # 你的 region_names 列表

with rasterio.open(tif_path) as src:
    world_region = src.read(1)
    b = src.bounds
extent = [b.left, b.right, b.bottom, b.top]

max_code = int(world_region.max())
cmap_list = [other_region_color] * (max_code + 1)
for code, col in zip(highlight_regions, highlight_colors):
    cmap_list[code] = col
# 透明化 "#N/A" 和 255
cmap_list[region_names.index("#N/A")] = (0,0,0,0)
if 255 <= max_code:
    cmap_list[255] = (0,0,0,0)
region_cmap = ListedColormap(cmap_list)

# -----------------------------------
# 修改后的绘图函数：添加国界边界
# -----------------------------------
def plot_heatmap_with_region(df, var, out_png):
    pivot = df.pivot(index='latitude', columns='longitude', values=var)
    pivot = pivot.sort_index(axis=0).sort_index(axis=1)
    data = pivot.values
    masked = np.ma.masked_equal(data, 0)
    cmap_hm = plt.cm.get_cmap('viridis_r').copy()
    cmap_hm.set_bad(color='none')

    lon_min, lon_max = pivot.columns.min(), pivot.columns.max()
    lat_min, lat_max = pivot.index.min(), pivot.index.max()

    fig, ax = plt.subplots(figsize=(32,16))
    ax.set_facecolor('white')

    # 1) region 底图
    ax.imshow(world_region,
              cmap=region_cmap,
              alpha=0.5,
              origin='upper',
              extent=extent,
              interpolation='nearest')

    # 2) region 边界线
    levels = np.arange(0.5, max_code+1, 1)
    ax.contour(world_region,
               levels=levels,
               colors='black',
               linewidths=0.2,
               origin='upper',
               extent=extent)

    # 3) 世界国界
    world_4326.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)

    # 4) 叠加 heatmap（0 值透明）
    # 设置色条范围：仅针对比例类变量使用 0-1 范围
    vmin, vmax = (0, 1) if var in ['pv_share', 'AE_share'] else (None, None)

    im = ax.imshow(masked,
                   cmap=cmap_hm,
                   alpha=1,
                   origin='lower',
                   extent=[lon_min, lon_max, lat_min, lat_max],
                   aspect='auto',
                   interpolation='nearest',
                   vmin=vmin,
                   vmax=vmax)

    # 5) legend for highlighted regions
    handles = [
        mpatches.Patch(color=cmap_list[code], label=f"{code}: {region_names[code]}")
        for code in highlight_regions
    ]
    ax.legend(handles=handles,
              title="Highlighted Regions",
              loc='lower left',
              bbox_to_anchor=(1.02, 0),
              borderaxespad=0,
              fontsize='small',
              title_fontsize='medium')

    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f"{var} Heatmap with Regions and Borders")

    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label(var)

    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    plt.savefig(out_png, bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f"Saved {out_png}")

# -----------------------------------
# 批量调用示例
# -----------------------------------
import pandas as pd
import numpy as np

# keys = ['main', 'HighPEM', 'LowAE', 'LowAE&HighPEM','LowPV','LowPV&HighPEM','LowPVAE','LowPV+AE&HighPEM',
#         'RegionalRE','RegionalRE_HighPEM','RegionalRE_LowAE','RegionalRE_LowAE&HighPEM']

keys = ['RegionRE_2030', 'RegionRE_LowAE', 'RegionRE_LowAE_2040', 'RegionRE_LowAE_2050']

# 原有变量 + 新增“光伏占比”
#variables = ['levelized_hydrogen_production_cost_per_kg', 'pv_share','AE_share']
variables = ['levelized_hydrogen_production_cost_per_kg']

for k in keys:
    df = pd.read_csv(f'DataOutput/Used_output/combined_{k}_hydrogen_production_results.csv')
    
    # 计算光伏在风电+光伏装机中的比例，避免除以零
    df['pv_share'] = df['pv_capacity'] / (df['pv_capacity'] + df['wind_capacity'])
    df['pv_share'] = df['pv_share'].fillna(0)
    
    df['AE_share'] = df['electrolyzer_capacity_AE'] / (df['electrolyzer_capacity_AE'] + df['electrolyzer_capacity_PEM'])
    df['AE_share'] = df['AE_share'].fillna(0)
    
    safe = k.replace('&','_and_')
    for var in variables:
        out_png = f'Mapping/New/{var}_{safe}_heatmap_with_region.png'
        plot_heatmap_with_region(df, var, out_png)



## 1.3 制氢结果绘图（其他参数）

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Function to plot a variable as a heatmap using imshow
def plot_variable_heatmap(file_path, variable, output_image_path):
    df = pd.read_csv(file_path)
    if 'latitude' in df.columns and 'longitude' in df.columns and variable in df.columns:
        lat_bins = np.linspace(df['latitude'].min(), df['latitude'].max(), 720)
        lon_bins = np.linspace(df['longitude'].min(), df['longitude'].max(), 1440)
        lat_centers = 0.5 * (lat_bins[:-1] + lat_bins[1:])
        lon_centers = 0.5 * (lon_bins[:-1] + lon_bins[1:])

        H, _, _ = np.histogram2d(
            df['latitude'], df['longitude'],
            bins=[lat_bins, lon_bins],
            weights=df[variable]
        )
        count, _, _ = np.histogram2d(
            df['latitude'], df['longitude'],
            bins=[lat_bins, lon_bins]
        )
        masked_data = np.divide(H, count, where=count != 0)

        plt.figure(figsize=(32, 16))
        plt.imshow(
            masked_data,
            cmap='viridis',
            origin='lower',
            extent=[lon_bins.min(), lon_bins.max(), lat_bins.min(), lat_bins.max()],
            aspect='auto',
            interpolation='bilinear'
        )
        plt.colorbar(label=variable)
        plt.xlabel('Longitude')
        plt.ylabel('Latitude')
        plt.title(f'{variable} Heatmap ({file_path.split("/")[-1].split(".")[0]})')
        plt.savefig(output_image_path, bbox_inches='tight', dpi=300)
        plt.close()
    else:
        print(f"Missing columns for {variable} in {file_path}")

# 定义所有要处理的 key 以及对应的文件前缀
keys = ['main', 'HighPEM', 'LowAE', 'LowAE&HighPEM']

# 定义所有要画的变量
variables = [
    'levelized_hydrogen_production_cost_per_kg',
    'wind_capacity',
    'pv_capacity',
    'battery_energy_capacity',
    'battery_storage_capacity',
    'electrolyzer_capacity_AE',
    'electrolyzer_capacity_PEM',
    'electrolyzer_utilization_hours_percent'
]

# 双重循环，自动生成输入/输出路径
for key in keys:
    input_csv = f'DataOutput/combined_{key}_hydrogen_production_results.csv'
    for var in variables:
        # 构建输出文件名，避免文件名中出现非法字符
        safe_key = key.replace('&', '_and_').replace('/', '_')
        output_png = f'Mapping/{var}_{safe_key}_heatmap_2.png'
        plot_variable_heatmap(input_csv, var, output_png)
        print(f"Saved {output_png}")


# 2. 筛选出来氨醇计算的格点

## 2.1 Wind Cover查看

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# ============================================
# 1. 读取 NetCDF 掩膜数据
# ============================================
nc_path = "E:/Workspace/Data_temp/wind_cover_global_masked.nc"
var_name = "Cover_wind_masked"

ds = xr.open_dataset(nc_path)

if var_name not in ds:
    raise ValueError(f"变量 '{var_name}' 不在文件中。可用变量：{list(ds.data_vars)}")

# ============================================
# 2. 经度从 0~360 转为 -180~180，并排序
# ============================================
# 转换坐标
ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
# 排序经度维度
ds = ds.sortby("longitude")

# 提取变量
mask = ds[var_name].values
lat = ds.latitude.values
lon = ds.longitude.values

print(f"掩膜 shape: {mask.shape}, 纬度数: {len(lat)}, 经度数: {len(lon)}")
print(f"经度范围: {lon.min()} ~ {lon.max()}")
print(f"纬度范围: {lat.min()} ~ {lat.max()}")

# ============================================
# 3. 绘制掩膜图像
# ============================================
plt.figure(figsize=(16, 8))
plt.imshow(
    mask,
    extent=[lon.min(), lon.max(), lat.min(), lat.max()],
    origin='lower',
    cmap='viridis',
    vmin=0,
    vmax=1,
    aspect='auto'
)
plt.colorbar(label=var_name)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"{var_name} (Longitude: -180 to 180)")
plt.grid(True)
plt.tight_layout()
plt.show()


## 2.2 根据Wind Cover筛选制氢格点 
(每个地区的前25%,并且在全球总体分布的前25%，至少保留10个点)

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import os

# 设置输入输出路径
input_folder = "DataOutput"
output_folder = "DataOutput/Filtered"
os.makedirs(output_folder, exist_ok=True)

# 掩膜加载 & 处理
mask_ds = xr.open_dataset("E:/Workspace/Data_temp/wind_cover_global_masked.nc")
mask = mask_ds["Cover_wind_masked"]
lat_vals = mask['latitude'].values
lon_vals = np.where(mask['longitude'].values > 180,
                    mask['longitude'].values - 360,
                    mask['longitude'].values)
lat_grid, lon_grid = np.meshgrid(lat_vals, lon_vals, indexing='ij')
mask_array = mask.values
mask_df = pd.DataFrame({
    "latitude": lat_grid.ravel(),
    "longitude": lon_grid.ravel(),
    "mask_value": mask_array.ravel()
})
mask_df["lat_round"] = mask_df["latitude"].round(5)
mask_df["lon_round"] = mask_df["longitude"].round(5)
mask_df = mask_df[mask_df["mask_value"] > 0]

# 通用筛选函数
def select_representative_low_cost_points(df, cost_col='levelized_hydrogen_production_cost_per_kg',
                                          region_col='region', min_keep=10, max_keep_ratio=0.25):
    selected_rows = []
    df[cost_col] = pd.to_numeric(df[cost_col], errors='coerce')
    global_threshold = df[cost_col].quantile(0.25)

    for region, group in df.groupby(region_col):
        if len(group) == 1:
            selected_rows.append(group)
            continue
        group_selected = group[group[cost_col] <= global_threshold]
        if group_selected.empty:
            group_final = group.nsmallest(min(min_keep, len(group)), cost_col)
        else:
            n_keep = max(min_keep, min(int(len(group) * max_keep_ratio), len(group_selected)))
            group_final = group_selected.nsmallest(n_keep, cost_col)
        selected_rows.append(group_final)
    return pd.concat(selected_rows, ignore_index=True)

# 全网格 DataFrame 构建（仅做一次）
lat_grid_all, lon_grid_all = np.meshgrid(lat_vals, lon_vals, indexing='ij')
full_grid = pd.DataFrame({
    "latitude": lat_grid_all.ravel(),
    "longitude": lon_grid_all.ravel()
})
full_grid["lat_round"] = full_grid["latitude"].round(5)
full_grid["lon_round"] = full_grid["longitude"].round(5)

# 遍历所有情景文件
for filename in os.listdir(input_folder):
    if not (filename.startswith("combined_") and filename.endswith("hydrogen_production_results.csv")):
        continue

    print(f"Processing: {filename}")
    scenario_name = filename.replace("combined_", "").replace("_hydrogen_production_results.csv", "")
    df = pd.read_csv(os.path.join(input_folder, filename))
    df["lat_round"] = df["latitude"].round(5)
    df["lon_round"] = df["longitude"].round(5)

    # 应用掩膜
    df_masked = df.merge(mask_df[["lat_round", "lon_round"]], on=["lat_round", "lon_round"], how="inner")
    df_masked.drop(columns=["lat_round", "lon_round"], inplace=True)

    # 找出掩膜覆盖不到的 region
    all_regions = set(df["region"].unique())
    masked_regions = set(df_masked["region"].unique())
    missing_regions = all_regions - masked_regions

    # 掩膜内筛选
    filtered_masked = select_representative_low_cost_points(df_masked)

    # 掩膜外回退筛选
    recovered_rows = []
    for region in missing_regions:
        group = df[df["region"] == region]
        if len(group) > 0:
            recovered_rows.append(group.nsmallest(10, "levelized_hydrogen_production_cost_per_kg"))

    filtered_df = pd.concat([filtered_masked] + recovered_rows, ignore_index=True)

    # 加入圆整坐标再补全全网格
    filtered_df["lat_round"] = filtered_df["latitude"].round(5)
    filtered_df["lon_round"] = filtered_df["longitude"].round(5)

    all_cols = df.columns.drop(['lat_round', 'lon_round'], errors='ignore')
    padded = full_grid.merge(
        filtered_df[["lat_round", "lon_round"] + list(all_cols)],
        on=["lat_round", "lon_round"],
        how="left"
    ).fillna(0)

    padded = padded.rename(columns={"lat_round": "latitude", "lon_round": "longitude"})
    padded = padded.drop(columns=["latitude_y", "longitude_y", "latitude_x", "longitude_x"], errors="ignore")

    # 保存
    output_file = os.path.join(output_folder, f"filtered_{scenario_name}.csv")
    padded.to_csv(output_file, index=False)
    print(f"Saved to: {output_file}")


## 2.3 形成格点集合

In [ ]:
import pandas as pd
import numpy as np
import os

# =========================
# 设置路径
# =========================
input_folder = "DataOutput/Filtered"
region_map_file = "DataOutput/RegionMapping.csv"
output_file = "DataOutput/union_representative_points.csv"

# =========================
# 加载 region 映射表
# =========================
region_map = pd.read_csv(region_map_file)
region_map.columns = ['region_code', 'region_name']

# =========================
# 收集所有代表点
# =========================
df_list = []

for filename in os.listdir(input_folder):
    if not (filename.startswith("filtered_") and filename.endswith(".csv")):
        continue

    file_path = os.path.join(input_folder, filename)
    df = pd.read_csv(file_path)

    # 只保留筛选出来的有效点
    df = df[df["levelized_hydrogen_production_cost_per_kg"] > 0]

    # 只保留必要列
    df = df[["latitude", "longitude", "region", "levelized_hydrogen_production_cost_per_kg"]]

    df_list.append(df)

# 合并所有情景点并去重（保留首个出现值）
combined_df = pd.concat(df_list).drop_duplicates(subset=["latitude", "longitude"])

# 匹配 region_name
combined_df = combined_df.merge(region_map, left_on="region", right_on="region_code", how="left")
combined_df.drop(columns=["region_code"], inplace=True)

# =========================
# 构建完整 0.25° 网格（1440×721）
# =========================
lat_vals = np.round(np.linspace(90, -90, 721), 5)
lon_vals = np.round(np.linspace(-180, 180 - 0.25, 1440), 5)
lat_grid, lon_grid = np.meshgrid(lat_vals, lon_vals, indexing='ij')

full_df = pd.DataFrame({
    "latitude": lat_grid.ravel(),
    "longitude": lon_grid.ravel()
})

# 合并所有格点，未命中的用 0 和 NaN 填充
merged = full_df.merge(combined_df, on=["latitude", "longitude"], how="left")
merged["levelized_hydrogen_production_cost_per_kg"] = merged["levelized_hydrogen_production_cost_per_kg"].fillna(0)
merged["region"] = merged["region"].fillna(0).astype(int)
merged["region_name"] = merged["region_name"].fillna("Unassigned")

# =========================
# 保存结果
# =========================
merged.to_csv(output_file, index=False)
print(f"已保存代表点并集数据（含 region 和 region_name）：{output_file}")


## 2.4 筛选后的点作图

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# -----------------------------------
# 边界矢量数据读取
# -----------------------------------
gdf = gpd.read_file("ne_110m_admin_0_countries.shp")
world_4326 = gdf.to_crs("EPSG:4326")

# -----------------------------------
# region 栅格 & colormap 配置（同你已有的）
# -----------------------------------
tif_path = "F:/Workspace/World_Region_025d.tif"
highlight_regions = [0,2,7,9,10,11,12,13,15,18,24,27,28,31]
highlight_colors  = [
    '#e41a1c','#377eb8','#4daf4a','#984ea3',
    '#ff7f00','#ffff33','#a65628','#f781bf','#00008b',
    '#999999','#66c2a5','#fc8d62','#8da0cb','#e78ac3'
]
other_region_color = '#DEEBF7'
region_names = [
    "EU-15","Canada","USA","Central America and Caribbean","Mexico",
    "Colombia","Southeast Asia","Australia_NZ","#N/A","Africa_Northern",
    "Africa_Western","Africa_Eastern","Africa_Southern","South Africa",
    "South America_Northern","South America_Southern","Brazil","Argentina",
    "European Free Trade Association","Russia","EU-12","Europe_Eastern",
    "Europe_Non_EU","Central Asia","Middle East","South Asia","Pakistan",
    "Japan","South Korea","India","Indonesia","China"
]  # 你的 region_names 列表

with rasterio.open(tif_path) as src:
    world_region = src.read(1)
    b = src.bounds
extent = [b.left, b.right, b.bottom, b.top]

max_code = int(world_region.max())
cmap_list = [other_region_color] * (max_code + 1)
for code, col in zip(highlight_regions, highlight_colors):
    cmap_list[code] = col
# 透明化 "#N/A" 和 255
cmap_list[region_names.index("#N/A")] = (0,0,0,0)
if 255 <= max_code:
    cmap_list[255] = (0,0,0,0)
region_cmap = ListedColormap(cmap_list)

# -----------------------------------
# 修改后的绘图函数：添加国界边界
# -----------------------------------
def plot_heatmap_with_region(df, var, out_png):
    pivot = df.pivot(index='latitude', columns='longitude', values=var)
    pivot = pivot.sort_index(axis=0).sort_index(axis=1)
    data = pivot.values
    masked = np.ma.masked_equal(data, 0)
    cmap_hm = plt.cm.get_cmap('viridis_r').copy()
    cmap_hm.set_bad(color='none')

    lon_min, lon_max = pivot.columns.min(), pivot.columns.max()
    lat_min, lat_max = pivot.index.min(), pivot.index.max()

    fig, ax = plt.subplots(figsize=(32,16))
    ax.set_facecolor('white')

    # 1) region 底图
    ax.imshow(world_region,
              cmap=region_cmap,
              alpha=0.5,
              origin='upper',
              extent=extent,
              interpolation='nearest')

    # 2) region 边界线
    levels = np.arange(0.5, max_code+1, 1)
    ax.contour(world_region,
               levels=levels,
               colors='black',
               linewidths=0.2,
               origin='upper',
               extent=extent)

    # 3) 世界国界
    world_4326.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)

    # 4) 叠加 heatmap（0 值透明）
    # 设置色条范围：仅针对比例类变量使用 0-1 范围
    vmin, vmax = (0, 1) if var in ['pv_share', 'AE_share'] else (None, None)

    im = ax.imshow(masked,
                   cmap=cmap_hm,
                   alpha=1,
                   origin='lower',
                   extent=[lon_min, lon_max, lat_min, lat_max],
                   aspect='auto',
                   interpolation='nearest',
                   vmin=vmin,
                   vmax=vmax)

    # 5) legend for highlighted regions
    handles = [
        mpatches.Patch(color=cmap_list[code], label=f"{code}: {region_names[code]}")
        for code in highlight_regions
    ]
    ax.legend(handles=handles,
              title="Highlighted Regions",
              loc='lower left',
              bbox_to_anchor=(1.02, 0),
              borderaxespad=0,
              fontsize='small',
              title_fontsize='medium')

    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f"{var} Heatmap with Regions and Borders")

    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label(var)

    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    plt.savefig(out_png, bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f"Saved {out_png}")

# -----------------------------------
# 批量调用示例
# -----------------------------------
import pandas as pd
import numpy as np

keys = ['main']
# 原有变量 + 新增“光伏占比”
#variables = ['levelized_hydrogen_production_cost_per_kg', 'pv_share','AE_share']
variables = ['levelized_hydrogen_production_cost_per_kg']

for k in keys:
    df = pd.read_csv(f'DataOutput/union_representative_points.csv')
    
    # 计算光伏在风电+光伏装机中的比例，避免除以零
#     df['pv_share'] = df['pv_capacity'] / (df['pv_capacity'] + df['wind_capacity'])
#     df['pv_share'] = df['pv_share'].fillna(0)
    
#     df['AE_share'] = df['electrolyzer_capacity_AE'] / (df['electrolyzer_capacity_AE'] + df['electrolyzer_capacity_PEM'])
#     df['AE_share'] = df['AE_share'].fillna(0)
    
    safe = k.replace('&','_and_')
    for var in variables:
        out_png = 'Mapping/Filter_heatmap_with_region.png'
        plot_heatmap_with_region(df, var, out_png)



# 3. 氨醇文件处理

## 3.1 给甲醇增加地区列
甲醇中仅有RegionRE的情景文件有地区列

In [ ]:
import os
import glob
import shutil
import pandas as pd

# 目录配置
meth_dir = "DataOutput/Output"
ref_dir = r"C:\Users\admin\POTSED\NoWater_valid_coords_segment"
output_dir = os.path.join(meth_dir, "methanol_withRegion")
os.makedirs(output_dir, exist_ok=True)

# 编号范围
start_idx = 0
end_idx = 1380
block_size = 60

for block_id in range(start_idx, end_idx + 1, block_size):
    ref_file = os.path.join(ref_dir, f"NoWater_valid_coords_with_region_{block_id}.csv")

    if not os.path.isfile(ref_file):
        print(f"⚠ 缺失参考文件：{ref_file}，跳过该 block")
        continue

    # 加载参考表
    ref = pd.read_csv(ref_file)

    # 标准化 region 列名
    if 'Region' in ref.columns:
        ref = ref.rename(columns={'Region': 'region'})
    elif 'region' not in ref.columns:
        print(f"❌ 错误：{ref_file} 缺少 'region' 列（Region 或 region）")
        continue

    # 检查必要列
    if 'latitude_index' not in ref.columns or 'longitude_index' not in ref.columns:
        print(f"❌ 错误：{ref_file} 缺少 latitude_index / longitude_index")
        continue

    # 查找该 block 下的所有 methanol 文件（包括变种）
    pattern = os.path.join(meth_dir, f"methanol_results_{block_id}*.csv")
    meth_files = glob.glob(pattern)

    if not meth_files:
        print(f"⚠ 没有找到匹配文件：{pattern}")
        continue

    for meth_file in meth_files:
        basename = os.path.basename(meth_file)
        out_path = os.path.join(output_dir, basename)

        # —— 对 RegionRE 场景直接复制，无需处理 ——  
        if basename.startswith(f"methanol_results_{block_id}_RegionRE"):
            shutil.copy(meth_file, out_path)
            print(f"🔄 直接复制 RegionRE 文件：{basename} → {out_path}")
            continue

        try:
            df = pd.read_csv(meth_file)

            if 'latitude' not in df.columns or 'longitude' not in df.columns:
                print(f"❌ 错误：{basename} 缺少 latitude 或 longitude 列")
                continue

            # —— 反算为索引 ——  
            df['latitude_index'] = 360 - df['latitude'] * 4
            df['longitude_index'] = (
                df['longitude'] * 4
                - block_id
                + (1440 if block_id >= 12 * 60 else 0)
            )

            # —— 合并 region ——  
            merged = pd.merge(
                df,
                ref[['latitude_index', 'longitude_index', 'region']],
                how='left',
                on=['latitude_index', 'longitude_index']
            )

            # 写入 region 列
            df['region'] = merged.get('region', pd.NA)

            # 统计未匹配数
            unmatched = df['region'].isna().sum()
            if unmatched > 0:
                print(f"⚠ {basename} 中有 {unmatched}/{len(df)} 行未匹配到 region")

            # 保存结果
            df.to_csv(out_path, index=False)
            print(f"✅ 已处理 {basename} → 保存至 {out_path}")

        except Exception as e:
            print(f"❌ 处理 {basename} 时出错：{e}")


## 3.2 拼接氨醇文件

In [ ]:
import os
import glob
import re
import pandas as pd
import numpy as np

# ----------------------------
# 配置
# ----------------------------
filter_value = 50000.0
products = {
    'hydrogen':  'levelized_hydrogen_production_cost_per_kg',
    #'ammonia':  'levelized_ammonia_cost_per_ton',
    #methanol': 'levelized_methanol_cost_per_ton',
    #'ftl':  'levelized_FTL_cost_per_ton'
}

# 全网格索引参数
lat_vals = np.arange(90, -90 - 0.25, -0.25)
lon_vals = np.arange(-180, 180, 0.25)

# 分块写的纬度块大小，可按内存情况调小（如 5、10、20…）
lat_chunk_size = 10

for prod, cost_col in products.items():
    # 匹配文件路径
    pattern = os.path.join(
        'DataOutput',
        f'{prod}_results_*_Region*.csv'
    )
    files = glob.glob(pattern)
    print(f"[{prod}] 共找到 {len(files)} 个文件")
    
    regex = re.compile(rf'{prod}_results_(\d+)(?:_(.+))?\.csv$')
    agg = {}  # suffix -> (sum_df, count_df)
    
    # —— 分文件累加 sum & count —— 
    for fn in files:
        base = os.path.basename(fn)
        m = regex.match(base)
        if not m:
            print(f"跳过：{base}")
            continue
        minute = int(m.group(1))
        suffix = m.group(2) or 'default'
        df = pd.read_csv(fn)
        
        # 经纬度转换 only ammonia
        if prod == 'ammonia':
            if 'latitude' in df:
                df['latitude'] = 90 - df['latitude'] * 0.25
            if 'longitude' in df:
                hour = minute // 60
                df['longitude'] = (
                    (hour * 60 + df['longitude']) / 4
                    - (360 if hour >= 12 else 0)
                )
        
        # 高成本过滤
        other_cols = [c for c in df.columns
                      if c not in ['latitude','longitude', cost_col]]
        mask = df[cost_col] > filter_value
        df.loc[mask, other_cols] = 0
        
        # groupby 求和&计数
        value_cols = other_cols + [cost_col]
        grp = df.groupby(['latitude','longitude'])[value_cols]
        sum_df, count_df = grp.sum(), grp.count()
        
        # 累加
        if suffix not in agg:
            agg[suffix] = (sum_df, count_df)
        else:
            prev_sum, prev_count = agg[suffix]
            agg[suffix] = (
                prev_sum.add(sum_df,   fill_value=0),
                prev_count.add(count_df, fill_value=0)
            )
    
    # —— 分块输出全网格 —— 
    for suffix, (sum_df, count_df) in agg.items():
        out_path = f'DataOutput/combined_{suffix}_{prod}_results.csv'
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        
        header_written = False
        # 按纬度块分批写
        for i in range(0, len(lat_vals), lat_chunk_size):
            block_lats = lat_vals[i:i+lat_chunk_size]
            # 构造这块的 MultiIndex
            idx_block = pd.MultiIndex.from_product(
                [block_lats, lon_vals],
                names=['latitude','longitude']
            )
            # 只在这块上 reindex 并算平均
            block_sum   = sum_df.reindex(idx_block,   fill_value=0)
            block_count = count_df.reindex(idx_block, fill_value=0)
            block_mean  = block_sum.divide(block_count).fillna(0)
            chunk_df    = block_mean.reset_index()
            
            # 写 CSV：首块写 header，其余追加
            chunk_df.to_csv(
                out_path,
                mode='w' if not header_written else 'a',
                header=not header_written,
                index=False
            )
            header_written = True
        
        print(f"✅ 已分块保存：{out_path}")


### 3.2.1 拼接中国氢氨醇文件

In [ ]:
import os
import glob
import re
import pandas as pd
import numpy as np

# ----------------------------
# 配置
# ----------------------------
filter_value = 500000.0
products = {
    #'hydrogen':  'levelized_hydrogen_production_cost_per_kg',
    'ammonia':  'levelized_ammonia_cost_per_ton',
    'methanol': 'levelized_methanol_cost_per_ton'
}

# 全网格索引参数
lat_vals = np.arange(90, -90 - 0.25, -0.25)
lon_vals = np.arange(-180, 180, 0.25)

# 分块写的纬度块大小，可按内存情况调小（如 5、10、20…）
lat_chunk_size = 10

for prod, cost_col in products.items():
    # 匹配文件路径
    pattern = os.path.join(
        'DataOutput',
        f'{prod}_results_China_*.csv'
    )
    files = glob.glob(pattern)
    print(f"[{prod}] 共找到 {len(files)} 个文件")
    
    regex = re.compile(rf'{prod}_results_China_(\d+)(?:_(.+))?\.csv$')
    agg = {}  # suffix -> (sum_df, count_df)
    
    # —— 分文件累加 sum & count —— 
    for fn in files:
        base = os.path.basename(fn)
        m = regex.match(base)
        if not m:
            print(f"跳过：{base}")
            continue
        minute = int(m.group(1))
        suffix = m.group(2) or 'default'
        df = pd.read_csv(fn)
        
        # # 经纬度转换 only ammonia
        # if prod == 'ammonia':
        #     if 'latitude' in df:
        #         df['latitude'] = 90 - df['latitude'] * 0.25
        #     if 'longitude' in df:
        #         hour = minute // 60
        #         df['longitude'] = (
        #             (hour * 60 + df['longitude']) / 4
        #             - (360 if hour >= 12 else 0)
        #         )
        
        # 高成本过滤
        other_cols = [c for c in df.columns
                      if c not in ['latitude','longitude', cost_col]]
        mask = df[cost_col] > filter_value
        df.loc[mask, other_cols] = 0
        
        # groupby 求和&计数
        value_cols = other_cols + [cost_col]
        grp = df.groupby(['latitude','longitude'])[value_cols]
        sum_df, count_df = grp.sum(), grp.count()
        
        # 累加
        if suffix not in agg:
            agg[suffix] = (sum_df, count_df)
        else:
            prev_sum, prev_count = agg[suffix]
            agg[suffix] = (
                prev_sum.add(sum_df,   fill_value=0),
                prev_count.add(count_df, fill_value=0)
            )
    
    # —— 分块输出全网格 —— 
    for suffix, (sum_df, count_df) in agg.items():
        out_path = f'DataOutput/combined_{suffix}_{prod}_China_results.csv'
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        
        header_written = False
        # 按纬度块分批写
        for i in range(0, len(lat_vals), lat_chunk_size):
            block_lats = lat_vals[i:i+lat_chunk_size]
            # 构造这块的 MultiIndex
            idx_block = pd.MultiIndex.from_product(
                [block_lats, lon_vals],
                names=['latitude','longitude']
            )
            # 只在这块上 reindex 并算平均
            block_sum   = sum_df.reindex(idx_block,   fill_value=0)
            block_count = count_df.reindex(idx_block, fill_value=0)
            block_mean  = block_sum.divide(block_count).fillna(0)
            chunk_df    = block_mean.reset_index()
            
            # 写 CSV：首块写 header，其余追加
            chunk_df.to_csv(
                out_path,
                mode='w' if not header_written else 'a',
                header=not header_written,
                index=False
            )
            header_written = True
        
        print(f"✅ 已分块保存：{out_path}")


## 3.3 氨醇绘图

In [ ]:
# -*- coding: utf-8 -*-
"""
NCC-style global heatmap with administrative borders & region overlay
- Clean white background, thin strokes, inward ticks
- Robust color scaling via percentile clipping (2–98%)
- Compact legend (multi-column), subtle region tints, crisp boundaries
- Publication-grade export (fonts kept as text in PDF/SVG)
"""

import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
from matplotlib.colors import ListedColormap, Normalize
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator

# =========================
# 1) NCC-like style preset
# =========================
def set_ncc_style(base=9):
    plt.rcParams.update({
        # fonts
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "mathtext.default": "regular",

        # axes
        "axes.facecolor": "white",
        "axes.edgecolor": "black",
        "axes.linewidth": 0.8,
        "axes.labelsize": base,
        "axes.titlesize": base + 2,
        "axes.titlepad": 8,

        # ticks
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
        "xtick.minor.size": 2,
        "ytick.minor.size": 2,
        "xtick.labelsize": base - 1,
        "ytick.labelsize": base - 1,

        # legend
        "legend.frameon": False,
        "legend.fontsize": base - 1,

        # save / figure
        "savefig.dpi": 300,
        "savefig.transparent": False,
        "savefig.bbox": "tight",
        "figure.dpi": 110,
        "figure.facecolor": "white",

        # keep text as text in PDF/SVG
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

set_ncc_style()

# =========================
# 2) Read boundaries / region raster
# =========================
gdf = gpd.read_file("ne_110m_admin_0_countries.shp")
world_4326 = gdf.to_crs("EPSG:4326")

tif_path = "F:/Workspace/World_Region_025d.tif"
with rasterio.open(tif_path) as src:
    world_region = src.read(1)
    b = src.bounds
extent = [b.left, b.right, b.bottom, b.top]

max_code = int(world_region.max())

# =========================
# 3) Region colormap
# =========================
highlight_regions = [0, 2, 7, 9, 10, 11, 12, 13, 15, 18, 24, 27, 28, 31]
highlight_colors  = [
    '#e41a1c','#377eb8','#4daf4a','#984ea3',
    '#ff7f00','#ffff33','#a65628','#f781bf','#00008b',
    '#999999','#66c2a5','#fc8d62','#8da0cb','#e78ac3'
]

region_names = [
    "EU-15","Canada","USA","Central America and Caribbean","Mexico",
    "Colombia","Southeast Asia","Australia_NZ","#N/A","Africa_Northern",
    "Africa_Western","Africa_Eastern","Africa_Southern","South Africa",
    "South America_Northern","South America_Southern","Brazil","Argentina",
    "European Free Trade Association","Russia","EU-12","Europe_Eastern",
    "Europe_Non_EU","Central Asia","Middle East","South Asia","Pakistan",
    "Japan","South Korea","India","Indonesia","China"
]

# base tint for non-highlighted regions (very light)
other_region_color = '#F2F6FB'

# construct cmap (ensure enough entries)
cmap_list = [other_region_color] * (max_code + 1)
for code, col in zip(highlight_regions, highlight_colors):
    if code <= max_code:
        cmap_list[code] = col

# make "#N/A" transparent if present in names and within code range
if "#N/A" in region_names:
    na_idx = region_names.index("#N/A")
    if na_idx <= max_code:
        cmap_list[na_idx] = (0, 0, 0, 0)

# also make code 255 transparent if present
if 255 <= max_code:
    cmap_list[255] = (0, 0, 0, 0)

region_cmap = ListedColormap(cmap_list)

# =========================
# 4) Helpers
# =========================
def robust_norm(masked_array, p_low=2, p_high=98):
    """Return Normalize with percentile-based vmin/vmax (ignores masked values)."""
    vals = np.asarray(masked_array).astype(float)
    if np.ma.isMaskedArray(masked_array):
        vals = masked_array.compressed()
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return Normalize()
    vmin, vmax = np.percentile(vals, [p_low, p_high])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
        vmin, vmax = np.nanmin(vals), np.nanmax(vals)
    return Normalize(vmin=vmin, vmax=vmax), (vmin, vmax)

def prettify_var(var):
    mapping = {
        "levelized_methanol_cost_per_ton": ("Levelized methanol cost", "USD per t"),
        "levelized_ammonia_cost_per_ton": ("Levelized ammonia cost", "USD per t"),
        "levelized_FTL_cost_per_ton": ("Levelized FTL cost", "USD per t"),
    }
    return mapping.get(var, (var, ""))

def format_cbar_label(title, unit):
    return f"{title}" + (f" ({unit})" if unit else "")

# =========================
# 5) Main plotting function
# =========================
def plot_heatmap_with_region(df, var, out_png):
    # Grid to 2D
    pivot = (df.pivot(index="latitude", columns="longitude", values=var)
               .sort_index(axis=0).sort_index(axis=1))

    # Mask zeros and non-finite
    data = np.array(pivot.values, dtype=float)
    data[~np.isfinite(data)] = np.nan
    data = np.ma.masked_invalid(data)
    # (可选) 若 0 代表无值，则也屏蔽
    data = np.ma.masked_where((data == 0) | (~np.isfinite(data)), data)

    lon_min, lon_max = float(pivot.columns.min()), float(pivot.columns.max())
    lat_min, lat_max = float(pivot.index.min()), float(pivot.index.max())

    # robust normalization
    norm, (vmin, vmax) = robust_norm(data, 2, 98)
    clipped_note = f"Color scale clipped at 2nd–98th percentiles [{vmin:.1f}, {vmax:.1f}]"

    # figure
    fig, ax = plt.subplots(figsize=(14, 7))  # 2:1 world aspect is often pleasing
    ax.set_facecolor("white")

    # (1) region tint background
    ax.imshow(world_region,
              cmap=region_cmap,
              alpha=0.42,               # subtle background tint
              origin="upper",
              extent=extent,
              interpolation="nearest",
              rasterized=True)

    # (2) region boundary lines (very thin)
    levels = np.arange(0.5, max_code + 1, 1)
    ax.contour(world_region,
               levels=levels,
               colors="#505050",
               linewidths=0.25,
               origin="upper",
               extent=extent)

    # (3) country borders (slightly thicker)
    world_4326.boundary.plot(ax=ax, edgecolor="black", linewidth=0.45)

    # (4) heatmap layer (on top)
    # viridis (not reversed)常用于成本“越高越热”；如需反转，可改为 'viridis_r'
    im = ax.imshow(data,
                   cmap="viridis",
                   norm=norm,
                   alpha=1.0,
                   origin="lower",
                   extent=[lon_min, lon_max, lat_min, lat_max],
                   aspect="auto",
                   interpolation="nearest",
                   rasterized=True)

    # axes cosmetics
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color("black")
    ax.tick_params(which="both", width=0.8)
    ax.xaxis.set_major_locator(MaxNLocator(6))
    ax.yaxis.set_major_locator(MaxNLocator(5))

    # labels & title
    vtitle, vunit = prettify_var(var)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"{vtitle} heatmap with regions and borders")

    # (5) legend for highlighted regions (multi-column, outside right)
    handles = []
    # for code in highlight_regions:
    #     if code <= max_code:
    #         name = region_names[code] if code < len(region_names) else f"Region {code}"
    #         color = cmap_list[code]
    #         if isinstance(color, tuple) and len(color) == 4 and color[3] == 0:
    #             continue  # skip fully transparent
    #         handles.append(mpatches.Patch(color=color, label=f"{code}: {name}"))

    if handles:
        # 自动分列：根据数量确定列数（最多3列）
        n = len(handles)
        ncol = 1 if n <= 8 else (2 if n <= 16 else 3)
        leg = ax.legend(handles=handles,
                        title="Highlighted regions",
                        loc="upper left",
                        bbox_to_anchor=(1.02, 1.0),
                        borderaxespad=0,
                        frameon=False,
                        handlelength=1.0,
                        handletextpad=0.5,
                        labelspacing=0.6,
                        ncol=ncol)
        if leg and leg.get_title():
            leg.get_title().set_fontweight("bold")

    # colorbar
    cbar = fig.colorbar(im, ax=ax, shrink=0.86, pad=0.015, aspect=30, extend="both")
    cbar.set_label(format_cbar_label(vtitle, vunit))
    cbar.ax.yaxis.set_major_locator(MaxNLocator(5))

    # clipped note（小注释，NCC 常见的图下注释风格）
    ax.text(0.0, -0.08, clipped_note,
            transform=ax.transAxes, fontsize=8, va="top", ha="left", color="#5A5A5A")

    # bounds and layout
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)

    # export
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    # 同时导出 PNG 与 PDF（论文常用矢量）
    base, _ = os.path.splitext(out_png)
    plt.savefig(out_png, dpi=600)
    plt.savefig(base + ".pdf")
    plt.close(fig)
    print(f"✔ Saved {out_png} & {base+'.pdf'}")

# =========================
# 6) Batch loop (kept same)
# =========================
input_dir = 'DataOutput/Used_output'
output_dir = 'Mapping/New'
vars_map = {
    'methanol': 'levelized_methanol_cost_per_ton',
    'ammonia': 'levelized_ammonia_cost_per_ton',
    'ftl': 'levelized_FTL_cost_per_ton'
}

pattern = os.path.join(input_dir, 'combined_*_results.csv')
for file_path in sorted(glob.glob(pattern)):
    fname = os.path.basename(file_path)
    if '_methanol_' in fname:
        prod = 'methanol'
    elif '_ammonia_' in fname:
        prod = 'ammonia'
    else:
        prod = 'ftl'

    var = vars_map[prod]
    scenario = fname.replace('combined_', '').replace(f'_{prod}_results.csv', '')
    df = pd.read_csv(file_path)
    safe = scenario.replace('&', '_and_')
    out_png = os.path.join(output_dir, f"{prod}_{safe}_{var}_heatmap.png")

    plot_heatmap_with_region(df, var, out_png)


## 3.4 氨醇图情景拼接

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# -----------------------------
# 配置：选择产品 & 目录
# -----------------------------
prod = 'methanol'   # 改为 'methanol' 可绘制甲醇图
var_map = {
    'ammonia': 'levelized_ammonia_cost_per_ton',
    'methanol': 'levelized_methanol_cost_per_ton'
}
var = var_map[prod]

input_dir = 'Mapping'  # 单幅图所在目录
output_file = f'{prod}_12scenarios_matrix.png'

# -----------------------------
# 手动指定要展示的 12 个场景（顺序即子图排布顺序）
# 示例场景名请替换为你自己文件名中的场景片段
# -----------------------------
if prod == 'ammonia':
    scenarios = [
        'None',
        'HighPEM',
        'LowAEHighPEM',
        'LowAE',
        'LowPV',
        'LowPV_and_HighPEM',
        'LowPVAE_and_HighPEM',
        'LowPVAE',
        'RegionalRE',
        'RegionalRE_HighPEM',
        'RegionalRE_LowAE_and_HighPEM',
        'RegionalRE_LowAE'
    ]
else:
    scenarios = [
        'None',
        'HighPEM',
        'LowAE&HighPEM',
        'LowAE',
        'LowPV',
        'LowPV_and_HighPEM',
        'LowPVAE_and_HighPEM',
        'LowPVAE',
        'RegionRE',
        'RegionRE_HighPEM',
        'RegionRE_LowAE_and_HighPEM',
        'RegionRE_LowAE'
    ]
# -----------------------------
# 画 3×4 矩阵
# -----------------------------
nrow, ncol = 3, 4
fig, axes = plt.subplots(nrow, ncol, figsize=(20, 10))
axes = axes.flatten()

for ax, scenario in zip(axes, scenarios):
    # 构造文件名
    safe = scenario.replace('&', '_and_')
    fname = f"{prod}_{safe}_{var}_heatmap.png"
    fpath = os.path.join(input_dir, fname)
    if not os.path.isfile(fpath):
        raise FileNotFoundError(f"找不到文件：{fpath}")
    
    img = mpimg.imread(fpath)
    ax.imshow(img)
    ax.axis('off')
    # 标题显示场景名
    ax.set_title(scenario, fontsize=10)

# 多余子图隐藏（理论上不会出现）
for ax in axes[len(scenarios):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"✔ 已保存：{output_file}")
